Import all the necessary libraries and initialize the Earth Engine API

In [1]:
import ee
import geemap
import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import mapping
from shapely.geometry import shape
import os
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
print("Folder :", os.getcwd())
os.chdir("E:/SynologyDrive/Articles/Data_paper/07 - DSM/")

Folder : e:\SynologyDrive\Articles\Data_paper\07 - DSM\script


Idenfy the Google Earth Engine (GEE) API account. Carreful the valdiation code is to be insert in the bar at the top of the notebook if run in Visual code

In [2]:
ee.Authenticate()
ee.Initialize()

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


Select the area of interest (AOI) by loading a shapefile and converting it to a GeoDataFrame and selection the parameters for the data selection.

In [8]:
# Local file
raster_path = r".\data\Large_grid.tif"
with rasterio.open(raster_path) as src:
    image = src.read(1)
    mask = image > 0     
    results = (
        {'properties': {'value': v}, 'geometry': s}
        for i, (s, v) in enumerate(
            shapes(image, mask=mask, transform=src.transform))
    )

# Convert in GeoDataFrame
geoms = list(results)
gdf = gpd.GeoDataFrame.from_features(geoms)

# Create a polygon and gee object
aoi_poly = gdf.union_all()
aoi = ee.Geometry(mapping(aoi_poly))

Plot the AOI

In [9]:
Map = geemap.Map()
Map.centerObject(aoi, 8)
Map.addLayer(aoi, {}, "AOI")
Map

Map(center=[37.001263030021775, 42.75000000000002], controls=(WidgetControl(options=['position', 'transparent_…

Compute the Sentinel-2 cloud mask function and bands extraction

In [10]:
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
        qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask).divide(10000)

dataset = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2021-01-01", "2022-01-31")  # period
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))  # cloud < 20%
    .map(maskS2clouds)
)

# Median
S2_composite = dataset.median().clip(aoi)

Compute the Landsat-8 cloud mask and bands extaction

In [11]:
def maskL8clouds(image):
    qa = image.select('QA_PIXEL')

    cloudBit        = 1 << 3
    cloudShadowBit  = 1 << 4
    # Cirrus have not been considered as it provides only black cells
    # cirrusConfMask  = 1 << 14

    mask = (qa.bitwiseAnd(cloudBit).eq(0)
            .And(qa.bitwiseAnd(cloudShadowBit).eq(0)))
    return image.updateMask(mask)



dataset = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA")
    .filterDate("2021-01-01", "2022-01-01")  # period
    .map(maskL8clouds)
)

# Median
L8_composite = dataset.median().clip(aoi)

Compute the MODIS bands extraction

In [12]:
modisLST = ee.ImageCollection("MODIS/061/MOD11A2")
modisLSTFiltered = modisLST.filterDate("2021-01-01", "2022-01-31").filterBounds(aoi)
modisrefl = ee.ImageCollection('MODIS/061/MOD09Q1')
modisreflFiltered = modisrefl.filterDate("2021-01-01", "2022-01-31").filterBounds(aoi)
modisvege = ee.ImageCollection("MODIS/061/MOD13Q1")
modisvegeFiltered = modisvege.filterDate("2021-01-01", "2022-01-31").filterBounds(aoi)

def kelvin_to_celsius(image):
    return image.multiply(0.02).subtract(273.15).rename('LST_Celsius')

lstDay = modisLSTFiltered.select('LST_Day_1km').map(kelvin_to_celsius)
lstDayMedian = lstDay.median().clip(aoi)

lstNight = modisLSTFiltered.select('LST_Night_1km').map(kelvin_to_celsius)
lstNightMedian = lstNight.median().clip(aoi)

redBand = modisreflFiltered.select('sur_refl_b01').median().clip(aoi)
nirBand = modisreflFiltered.select('sur_refl_b02').median().clip(aoi)
NDVIBand = modisvegeFiltered.select('NDVI').median().clip(aoi)
EVIBand = modisvegeFiltered.select('EVI').median().clip(aoi)

Compute the Potential Evapotranspiration (PET) using the Global Aridity and PET Database

In [14]:
# PET
et_monthly = ee.ImageCollection("projects/sat-io/open-datasets/global_et0/global_et0_monthly")

# Select all the months
months = ['et0_V3_01','et0_V3_02','et0_V3_03','et0_V3_04','et0_V3_05','et0_V3_06','et0_V3_07','et0_V3_08',
          'et0_V3_09','et0_V3_10','et0_V3_11','et0_V3_12']
subset = et_monthly.filter(ee.Filter.inList('system:index', months))

# Sum of PET
pet_select = subset.sum().rename('ET0_sum').clip(aoi)

Add the DEM and the Landuse (from ESRI) layers

In [15]:
# DEM
dem = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM").mosaic().clip(aoi)

Landuse = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi)


Export the DEM Landuses, MODIS and other as a local image and the Landsat and Sentinel as images to Google Drive, as they are to heavy to be exported locally.

In [ ]:
geemap.ee_export_image(
    Landuse,
    filename=r".\data\Others\Landuse_raw.tif",
    scale=30,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    pet_select,
    filename=r".\data\Others\PET_raw.tif",
    scale=800,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    lstDayMedian,
    filename=r".\data\MODIS\MODIS_LST_day_raw.tif",
    scale=1000,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    lstNightMedian,
    filename=r".\data\MODIS\MODIS_LST_night_raw.tif",
    scale=1000,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    redBand,
    filename=r".\data\MODIS\MODIS_Red_raw.tif",
    scale=250,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    nirBand,
    filename=r".\data\MODIS\MODIS_NIR_raw.tif",
    scale=250,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    NDVIBand,
    filename=r".\data\MODIS\MODIS_NDVI_raw.tif",
    scale=250,
    region=aoi,
    crs="EPSG:32638"
)

geemap.ee_export_image(
    EVIBand,
    filename=r".\data\MODIS\MODIS_EVI_raw.tif",
    scale=250,
    region=aoi,
    crs="EPSG:32638"
)

task = ee.batch.Export.image.toDrive(
        image=dem,
        description = f"DEM_raw",
        scale = 30,
        region = aoi,
        crs = 'EPSG:32638',
        folder = 'GoogleEarthEngine',
        maxPixels = 1e13
    )
task.start()

S2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B11', 'B12']

for band in S2_bands:
    task = ee.batch.Export.image.toDrive(
        image=S2_composite.select([band]),
        description = f"Sentinel2_{band}_2021_MedianComposite",
        scale = 30,
        region = aoi,
        crs = 'EPSG:32638',
        folder = 'GoogleEarthEngine',
        maxPixels = 1e13
    )
    task.start()

L8_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8']

for band in L8_bands:
    task = ee.batch.Export.image.toDrive(
        image=L8_composite.select([band]),
        description = f"Landsat8_{band}_2021_MedianComposite",
        scale = 30,
        region = aoi,
        crs = 'EPSG:32638',
        folder = 'GoogleEarthEngine',
        maxPixels = 1e13
    )
    task.start()

Generating URL ...
An error occurred while downloading.
Total request size (86354475 bytes) must be less than or equal to 50331648 bytes.
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\07 - DSM\data\Others\Landuse_raw.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\07 - DSM\data\Others\PET_raw.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\07 - DSM\data\MODIS\MODIS_LST_day_raw.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\07 - DSM\data\MODIS\MODIS_LST_night_raw.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\07 - DSM\data\MODIS\MODIS_Red_raw.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data_paper\07 - DSM\data\MODIS\MODIS_NIR_raw.tif
Generating URL ...
Please wait ...
Data downloaded to E:\SynologyDrive\Articles\Data

Vizualize the different bands and DEM

In [16]:
Map = geemap.Map()

# Vizualization parameters
dem_vis = {'min': 0, 'max': 4000, 'palette': ['blue', 'green', 'brown', 'white']}
PET_vis = {'min': 1140, 'max': 1510, 'palette': ['yellow', 'green', 'darkgreen']}
LST_day_vis = {'min': 18, 'max': 38, 'palette': ['white', 'blue']}
LST_night_vis = {'min': 6, 'max': 18.5, 'palette': ['white', 'blue']}
Red_vis = {'min': 18, 'max': 3000, 'palette': ['white', 'red']}
NIR_vis = {'min': -21, 'max': 4080, 'palette': ['white', 'red']}
NDVI_vis = {'min': -1200, 'max': 5340, 'palette': ['yellow', 'green', 'darkgreen']}
EVI_vis = {'min': -400, 'max': 3600, 'palette': ['white', 'blue']}
Landuse_vis = {'bands': ['Map']}

# Add layers
Map.centerObject(aoi, 9)
Map.addLayer(S2_composite.select(["B4", "B3", "B2"]), {"min": 0, "max": 0.3}, "Sentinel 2 Median")
Map.addLayer(L8_composite.select(["B4", "B3", "B2"]), {"min": 0, "max": 0.3}, "Landsat 8 Median")
Map.addLayer(Landuse, Landuse_vis, "Landuse from ESRI")
Map.addLayer(dem, dem_vis, "DEM GLO 30")
Map.addLayer(pet_select, PET_vis, "PET Mars-August")
Map.addLayer(lstDayMedian, LST_day_vis, "LST Day Median")
Map.addLayer(lstNightMedian, LST_night_vis, "LST Night Median")
Map.addLayer(redBand, Red_vis, "MODIS Red Band Median")
Map.addLayer(nirBand, NIR_vis, "MODIS NIR Band Median")
Map.addLayer(NDVIBand, NDVI_vis, "MODIS NDVI Median")
Map.addLayer(EVIBand, EVI_vis, "MODIS EVI Median")

# plot map
Map

Map(center=[37.001263030021775, 42.75000000000002], controls=(WidgetControl(options=['position', 'transparent_…

Image will be dowloaded localy from ther R code